# Bear Classifier

Upload a bear image and click **Classify** to predict whether it is a black bear, grizzly bear, or teddy bear.

In [ ]:
from fastai.vision.all import *
from fastai.vision.widgets import *
from io import BytesIO

model_paths = [Path('export.pkl'), Path('bears/export.pkl')]
model_path = next((p for p in model_paths if p.exists()), None)
learn_inf = load_learner(model_path) if model_path is not None else None

btn_upload = widgets.FileUpload(accept='image/*', multiple=False)
btn_run = widgets.Button(description='Classify', button_style='primary')
out_pl = widgets.Output()
lbl_pred = widgets.Label()

if learn_inf is None:
    lbl_pred.value = "Model file export.pkl not found. Run learn.export() and add export.pkl to this repository."
else:
    lbl_pred.value = "Upload an image, then click Classify."

def _uploaded_content(upload):
    if getattr(upload, 'value', None):
        value = upload.value
        if isinstance(value, dict):
            item = next(iter(value.values()))
            return item.get('content', item)
        if isinstance(value, (tuple, list)):
            item = value[-1]
            return item.get('content', item) if isinstance(item, dict) else item
    if getattr(upload, 'data', None):
        return upload.data[-1]
    return None

def _make_image(content):
    if isinstance(content, memoryview):
        content = content.tobytes()
    if isinstance(content, (bytes, bytearray)):
        return PILImage.create(BytesIO(content))
    return PILImage.create(content)

def on_click_classify(change):
    if learn_inf is None:
        return

    content = _uploaded_content(btn_upload)
    if content is None:
        lbl_pred.value = 'Please upload an image first.'
        return

    img = _make_image(content)
    out_pl.clear_output(wait=True)
    with out_pl:
        display(img.to_thumb(256, 256))

    pred, pred_idx, probs = learn_inf.predict(img)
    lbl_pred.value = f'Prediction: {pred}; Probability: {probs[pred_idx]:.04f}'

btn_run.on_click(on_click_classify)

In [ ]:
VBox([
    widgets.Label('Select your bear image:'),
    btn_upload,
    btn_run,
    out_pl,
    lbl_pred,
])